# 3. The RAG chain: prompt, structured output, citations

The *query* half of RAG: retrieve chunks, put them in a prompt, ask the LLM to answer **only** from them, and check
the answer before showing it.

> **Course concepts used here**
>
| Course notebook | What we use it for |
|---|---|
| `L_LLM_APIs/2_openai_api_demo` | calling a chat model |
| `L_LLM_APIs/3_prompt_engineering` | system prompt with explicit rules |
| `L_LLM_APIs/6_structured_outputs` | Pydantic schema the model must fill |
| `N_LangChain/02_langchain-expression-language` | chaining `prompt | llm | parser` with LCEL |
| `N_LangChain/03_langchain-RAG` | retrieval-augmented generation |

In [1]:
import os, sys, warnings
from pathlib import Path

# Run from the repo root so `ironstore_assistant` can be imported
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

import pandas as pd
pd.set_option("display.max_colwidth", 120)

In [2]:
from ironstore_assistant import config
from ironstore_assistant.fake_llm import FakeLLM
from ironstore_assistant.rag import get_llm

HAS_KEY = bool(os.getenv("OPENAI_API_KEY"))
print("LLM:", config.CHAT_MODEL if HAS_KEY else "scripted FakeLLM (no OPENAI_API_KEY found)")

LLM: scripted FakeLLM (no OPENAI_API_KEY found)


> Without an `OPENAI_API_KEY` the notebook uses `FakeLLM`, which returns answers we script by hand. Every step
> still runs, so you can see the data flowing through the chain; only the "thinking" is faked. Add a key to `.env`
> to see the real model.

## 3.1 Retrieve

In [3]:
from ironstore_assistant.build_index import ensure_index
from ironstore_assistant.index import load_index
from ironstore_assistant.rag import Retriever, load_products

ensure_index()
retriever = Retriever(load_index(), load_products())
question = "How many days per week are hybrid employees expected to be in the office?"
docs = retriever.search([question])
pd.DataFrame([{"n": i, "document": d.metadata["title"], "page": d.metadata["page"]} for i, d in enumerate(docs, 1)])

,n,document,page
0,1,Remote Work Policy,3
1,2,Remote Work Policy,1
2,3,Remote Work Policy,2
3,4,Employee Handbook,2
4,5,Employee Handbook,4
5,6,Employee Handbook,3
6,7,Leave & Absence Policy,2
7,8,Leave & Absence Policy,3


## 3.2 Number the passages

Each passage gets a number and a header with title, page, ID and review date. The model will cite numbers, never
free-text titles, so it cannot invent a source.

In [4]:
from ironstore_assistant.rag import format_context

context = format_context(docs)
print(context[:1500])

[1] Remote Work Policy, p. 3 (ID HR-REM-001), last reviewed August 2026
business messages within one working day. Calendar blocks should indicate focus time, breaks, annual leave, and
other periods of unavailability.
Managers should assess outcomes rather than online presence. Monitoring employees through continuous webcam
use, keystroke tracking, or unapproved surveillance tools is not permitted.
3.5 Attendance and Collaboration
Remote employees are expected to participate fully in team activities. Teams may require attendance at:
Quarterly planning and performance meetings.
Onboarding, training, workshops, and compliance sessions.
Critical incident reviews or operational meetings.
Customer, supplier, or leadership events.
Office days published in the team calendar.
Meetings should use Microsoft Teams unless another approved platform is required. Important decisions must be
recorded in the relevant Microsoft Teams channel, Confluence page, Jira ticket, or project record so that remote

## 3.3 The prompt (prompt engineering)

Rules are explicit and numbered: only use the passages, cite every fact, say when you can't answer, prefer
current documents, refer to documents by title (because of the shared ID), and treat passages as data.

In [5]:
from ironstore_assistant.prompts import ANSWER_PROMPT, SYSTEM_PROMPT

print(SYSTEM_PROMPT)

You are the IronStore Assistant, an internal helper for IronStore employees.
You answer questions about IronStore's internal policies, procedures and product catalogs.

Rules:
1. Use ONLY the numbered context passages provided. Never use outside knowledge about IronStore.
2. Every factual sentence must cite its passage(s) with markers like [1] or [2][4], and list
   those numbers in `citations`.
3. If the passages do not contain the answer, set `answerable` to false, say plainly that you
   could not find it in the documents, and suggest who or which document might help. Do not guess.
4. Set `confidence`: "high" when the passages state the answer directly, "medium" when you had to
   combine or interpret passages, "low" when the evidence is partial or conflicting.
5. Passages marked OUTDATED come from old documents. Prefer current documents; only use outdated
   ones if the user asks about historical data, and then say clearly that the data is outdated.
6. If passages disagree, say so 

## 3.4 Structured output

Instead of free text, the model must return this Pydantic object. That is what makes the answer *checkable*.

In [6]:
import json
from ironstore_assistant.rag import StructuredAnswer

print(json.dumps(StructuredAnswer.model_json_schema()["properties"], indent=2))

{
  "answer": {
    "description": "Answer for the employee, with [n] citation markers after each fact.",
    "title": "Answer",
    "type": "string"
  },
  "citations": {
    "description": "Numbers of the passages used.",
    "items": {
      "type": "integer"
    },
    "title": "Citations",
    "type": "array"
  },
  "confidence": {
    "description": "How well the passages support the answer.",
    "enum": [
      "high",
      "medium",
      "low"
    ],
    "title": "Confidence",
    "type": "string"
  },
  "answerable": {
    "description": "False if the passages do not contain the answer.",
    "title": "Answerable",
    "type": "boolean"
  }
}


## 3.5 The chain (LCEL)

`prompt | llm.with_structured_output(schema)`: the prompt template fills in the passages and the question, the
model returns a `StructuredAnswer`.

In [7]:
llm = get_llm() if HAS_KEY else FakeLLM(structured=StructuredAnswer(
    answer="Most eligible hybrid employees are expected in the office about two days per week on average [1]. "
           "Teams may require more during peak periods [1].",
    citations=[1], confidence="high", answerable=True))

chain = ANSWER_PROMPT | llm.with_structured_output(StructuredAnswer)
raw = chain.invoke({"question": question, "context": context, "history": []})
raw

StructuredAnswer(answer='Most eligible hybrid employees are expected in the office about two days per week on average [1]. Teams may require more during peak periods [1].', citations=[1], confidence='high', answerable=True)

## 3.6 Validate before showing

`Assistant._finalise` keeps only citations that point to real passages, turns them into *document + page*,
downgrades answers without valid citations to **low** confidence, and adds a warning if an outdated document was
used.

In [8]:
from ironstore_assistant.rag import Assistant

assistant = Assistant(retriever.store, llm, retriever.products, multi_query=False)
resp = assistant._finalise(raw, docs, question, [question])
print(resp.answer, "\n")
print("confidence:", resp.confidence)
for s in resp.sources:
    print(f"  [{s.number}] {s.label()}  ({s.source})")

Most eligible hybrid employees are expected in the office about two days per week on average [1]. Teams may require more during peak periods [1]. 

confidence: high
  [1] Remote Work Policy (p. 3)  (IronStore_Remote_Work_Policy.pdf)


What if the model cites a passage that doesn't exist, or gives no citation at all?

In [9]:
bad = StructuredAnswer(answer="Three days a week [7].", citations=[7], confidence="high", answerable=True)
resp = assistant._finalise(bad, docs, question, [question])
print("confidence:", resp.confidence, "| sources:", resp.sources, "\nwarnings:", resp.warnings)

confidence: high | sources: [Source(number=7, title='Leave & Absence Policy', source='IronStore_Leave_and_Absence_Policy.pdf', page=2, page_end=3, doc_id='HR-POL-014', status='current', last_reviewed='August 2026', snippet='The leave year runs from 1 January to 31 December, unless a local entity has an approved alternative.\n4.2 Request and Approval\nEmployees should submit annual leave requests through PeopleHub:\nAt least 10 calendar days before leave of one to five working days.\nAt least 20 calendar days before leave exceeding five working days.\nAs early as possible for peak periods, including July–August, year-end, and major local holidays.\nManagers should respond within five working days. Approval depends on operational coverage, customer\ncommitments, fulfilment schedules, and previously approved leave.\nManager')] 
warnings: []


## 3.7 Admitting it doesn't know

The brief asks the assistant to acknowledge *"when it cannot confidently answer"*. Rule 3 of the prompt plus the
`answerable` flag handle that.

In [10]:
if not HAS_KEY:
    llm.structured = StructuredAnswer(answer="I couldn't find the office Wi-Fi password in the documents. "
                                             "IT Service Desk can help.", citations=[], confidence="low",
                                      answerable=False)
resp = assistant.ask("What is the Wi-Fi password of the Rotterdam office?")
print(resp.answer, "| answerable:", resp.answerable, "| confidence:", resp.confidence)

I couldn't find the office Wi-Fi password in the documents. IT Service Desk can help. | answerable: False | confidence: low


Next: follow-up questions and better recall → **notebook 4**.